# 03 · Multimodal Experience —— 视觉编码器+投影层+LLM，看图说话 toy

**家族位置**：`06_Transformer_Vision_Multimodal` 最后一站。01 训出 ViT（监督版），02 证明对比学习饿数据，本章把 ViT 当**眼睛**、拼一座**字符级 LLM** 作嘴——这就是 LLaVA 三段式（vision → projector → LLM）的 toy 复刻。10 类模板描述句半合成数据，CPU 分钟级可跑。

**学习目标**
1. 三段式架构：冻结 ViT + 两层投影 + 因果 MiniLLM，看图逐字生成模板描述句
2. toy 数据是已知短板的诚实版：模板描述句 vs 真实 caption
3. 图文对齐+生成双实验：描述句续写 + 图像前缀注意力热力
4. 06 家族收官：ViT（眼睛）→ CLIP（相亲）→ LLaVA（眼睛接大脑）

## 1. 原理：眼睛、视神经、嘴

### 通俗理解

**一句话**：01 的 ViT 是“哑巴眼睛”——看得见（分类 0.36）但说不出；本章给它接一条**视神经**（投影层，把 128 维图向量翻成 LLM 能读的“方言”）+ 一张**嘴**（字符级因果 LLM），让它看着图逐字吐出 `"a photo of a ship sailing on the sea."`。

**比喻**：LLaVA 像给盲人装眼睛——ViT 把像素变成 128 维“视觉印象”，投影层翻译成 LLM 母语，LLM 像鹦鹉一样按印象续写。真 LLaVA 用 13B LLM + 真实 caption；toy 版用 3 层小 LLM + 10 句模板，**只学“按图选哪句模板”，不学“自由写作”**——这是诚实版，不是骗人版。

### 结构账

```
眼睛： ViT-Tiny(dim128/L4) 监督预热 3ep（本章内训，复用 01 配方），训完冻结
视神经： projector 128→128→128×n_img(4)，把 CLS 向量翻成 4 个“图像词”
嘴： MiniLLM dim128/depth3/heads4，因果自注意力，字符词表 42，max_len 80
输入： [IMG×4, <s>, a photo of a ship sailing on the sea., </s>] teacher forcing 预测下一字
推理： [IMG×4, <s>] 自回归续写到 </s>
```

- **与真 LLaVA 的对应**：同三段（frozen vision + trainable projector + LLM）；不同在 LLM 是 3 层玩具、文本是 10 句模板
- **评估**：描述句生成 acc（整句全对）+ 逐字 PPL + 类名命中 + 图像前缀注意力热力

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "common").exists():
    ROOT = ROOT.parent
assert (ROOT / "common").exists(), "向上未找到 common 目录"
sys.path.insert(0, str(ROOT))

from common.data import load_cifar10_local, captions_from_labels, CIFAR10_CLASSES
from common.models import ViTTiny, MiniLLM, tokenize_texts
from common.engine import fit
from common.utils import set_seed, setup_chinese_font, count_params, unnormalize

set_seed(0)
setup_chinese_font()
FIGS = Path.cwd() / "figs"
FIGS.mkdir(exist_ok=True)
print("torch:", torch.__version__)

N_TRAIN, N_VAL = 3000, 600
EPOCHS_CLS, EPOCHS_CAP, BATCH, LR = 3, 10, 128, 3e-4
MAXC = 64
Xtr, ytr, Xva, yva, Xte, yte = load_cifar10_local(N_TRAIN, N_VAL, seed=0)
cap_tr = captions_from_labels(ytr)
cap_va = captions_from_labels(yva)
cap_te = captions_from_labels(yte)
print(f"train {tuple(Xtr.shape)} / test {tuple(Xte.shape)} | 例 '{cap_tr[0]}' (len {len(cap_tr[0])})")


## 2. 数据：10 类 × 1 句模板（已知短板，先说清楚）

每类 1 句固定模板（如 ship → `a photo of a ship sailing on the sea.`），共 10 句。**这不是真实 caption**：真实 caption 同一张图可以有 5 种说法、背景/动作/计数都不同；模板把“看图说话”降级成“按图选 10 选 1 再背出来”。本章诚实承认：**只证明三段式管道跑得通，不证明学会了开放描述**。

In [ ]:
cap_ids_tr = tokenize_texts(cap_tr, max_len=MAXC)
cap_ids_va = tokenize_texts(cap_va, max_len=MAXC)
cap_ids_te = tokenize_texts(cap_te, max_len=MAXC)
print(f"tokenized {tuple(cap_ids_tr.shape)} 词表 {len(MiniLLM(ViTTiny()).vocab)} | 最长句 {max(len(s) for s in cap_tr)} 字符")
# fig0：10 类模板全览
fig, ax = plt.subplots(figsize=(8.6, 3.4))
ax.axis("off")
for j, name in enumerate(CIFAR10_CLASSES):
    from common.data import CAPTION_TMPL
    ax.text(0.02, 0.94-j*0.093, f"{name:10s} → {CAPTION_TMPL[name]}", fontsize=8, family="monospace", transform=ax.transAxes)
ax.set_title("模板描述句：10 类 × 1 句（toy 短板：同类千图一句，真实 caption 应百花齐放）", fontsize=10)
plt.tight_layout()
plt.savefig(FIGS / "fig0_templates.png", dpi=150, bbox_inches="tight")
plt.show()


## 3. 第一段：ViT 眼睛预热 3ep（监督分类，训完冻结）

In [ ]:
train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=BATCH, shuffle=True)
val_loader = DataLoader(TensorDataset(Xva, yva), batch_size=512)
torch.manual_seed(0)
vit = ViTTiny(dim=128, depth=4, heads=4)
hist_vit = fit(vit, train_loader, val_loader, epochs=EPOCHS_CLS, lr=LR)
print(f"ViT params={count_params(vit)} best-val={max(hist_vit['val_acc']):.4f}")

# fig1：ViT 预热曲线
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(hist_vit["val_acc"], marker="o", color="#4C72B0", label="val acc")
ax.plot(hist_vit["train_acc"], marker="s", color="#DD8452", label="train acc")
ax.set_xlabel("epoch"); ax.set_ylabel("acc")
ax.set_title("眼睛预热 3ep（监督分类，为 caption 提供图像语义）")
ax.legend()
plt.tight_layout()
plt.savefig(FIGS / "fig1_vit.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. 第二段：投影+LLM 学“按图背模板”（10ep，冻结眼睛）

In [ ]:
model = MiniLLM(vision=vit, llm_dim=128, llm_depth=3, heads=4, img_dim=128, n_img_tokens=4, max_len=96)
print(f"MiniLLM 可训 params={count_params(model)} (视觉塔已冻结)")
cap_loader = DataLoader(TensorDataset(Xtr, cap_ids_tr), batch_size=BATCH, shuffle=True)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=3e-4, weight_decay=0.05)
hist, ppls = [], []
n = len(cap_loader.dataset)
for ep in range(1, EPOCHS_CAP+1):
    model.train()
    tot = 0.0
    for xb, cb in cap_loader:
        loss = model.caption_loss(xb, cb)
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item()*len(xb)
    avg = tot/n
    hist.append(avg); ppls.append(float(np.exp(min(avg, 8))))
    print(f"epoch {ep:02d} | cap CE {avg:.4f} | PPL {np.exp(min(avg,8)):.2f}", flush=True)

# fig2：caption CE + PPL 双曲线
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].plot(hist, marker="o", color="#4C72B0")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("caption CE")
axes[0].set_title("投影+LLM 学习曲线（冻结眼睛）")
axes[1].plot(ppls, marker="s", color="#55A868")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("逐字 PPL")
axes[1].set_title("困惑度下降 = 逐字越猜越准")
plt.tight_layout()
plt.savefig(FIGS / "fig2_cap.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. 生成体验：看图说话 + 类名命中 acc

In [ ]:
model.eval()
DEM = 200
with torch.no_grad():
    out = model.generate(Xte[:DEM])
gens = [model.decode(row) for row in out]
# 整句全对 + 类名命中
trues = cap_te[:DEM]
exact = np.mean([g == t for g, t in zip(gens, trues)])
def hit(g, y):
    return CIFAR10_CLASSES[int(y)] in g
classhit = np.mean([hit(g, y) for g, y in zip(gens, yte[:DEM])])
print(f"生成整句全对(200张)={exact:.4f} | 类名命中={classhit:.4f}")
for k in range(3):
    print(f"  图{k}: true='{trues[k]}'\n       gen ='{gens[k]}'")

# fig3：4 张看图说话条带
fig, axes = plt.subplots(1, 4, figsize=(10, 3.0))
idxs = [0, 1, 2, 3]
for ax, i in zip(axes, idxs):
    ax.imshow(unnormalize(Xte[i]))
    ok = gens[i] == trues[i]
    ax.set_title(f"true类 {CIFAR10_CLASSES[int(yte[i])]}\n{'✓' if ok else '✗'} {gens[i][:32]}...", fontsize=7, color="#1E8449" if ok else "#C0392B")
    ax.axis("off")
plt.suptitle(f"看图说话抽样（整句全对 {exact:.2f}，类名命中 {classhit:.2f}）——模板背诵机", fontsize=11)
plt.tight_layout()
plt.savefig(FIGS / "fig3_gen.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. 图像前缀注意力：嘴在生成时看不看眼睛？

In [ ]:
# fig4：生成第 1 个词时，最后一层对 [IMG×4 + 已生成] 的注意力（头平均）
model.eval()
with torch.no_grad():
    logits, attns = model(Xte[:1], cap_ids_te[:1][:, :-1], return_attn=True)
    last = attns[-1][0].mean(dim=0).cpu().numpy()  # (S,S)
    # 行=最后一个文本位，列=前 4 个 IMG 位
    img_cols = last[-1, :4]
fig, axes = plt.subplots(1, 2, figsize=(8, 3.2))
axes[0].imshow(unnormalize(Xte[0]))
axes[0].set_title(f"输入图：{CIFAR10_CLASSES[int(yte[0])]}", fontsize=10)
axes[0].axis("off")
axes[1].bar(range(4), img_cols, color="#4C72B0")
axes[1].set_xticks(range(4)); axes[1].set_xticklabels(["IMG0","IMG1","IMG2","IMG3"])
axes[1].set_ylabel("注意力权重（生成末位→4个图像词）")
axes[1].set_title("嘴→眼睛的注意力：>0.25 均值即图像前缀被利用")
for i, v in enumerate(img_cols):
    axes[1].text(i, v+0.01, f"{v:.3f}", ha="center", fontsize=8)
plt.tight_layout()
plt.savefig(FIGS / "fig4_imgattn.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"IMG 注意力 {img_cols.round(3)} 均值 {img_cols.mean():.3f}（均匀基线 0.25/总长归一后约 {4/last.shape[1]:.3f}）")


## 7. 总结（06 家族收官）

**本项目收获**

1. LLaVA 三段式 toy 闭环：冻结 ViT（3ep 预热）+ 投影 4 词 + 3 层因果 LLM，10ep 学会按图背 10 句模板
2. 诚实短板：模板=10 选 1 背诵，不是开放 caption；整句 acc 高不代表理解力强，类名命中才是可用口径
3. 图像前缀注意力热力：生成时嘴确实在看眼睛（>均匀基线即利用）
4. 06 家族收官：01 眼睛（ViT 切块+CLS）→ 02 相亲（CLIP 双塔+InfoNCE）→ 03 眼睛接大脑（投影+LLM 生成）

**06 家族一句话**：把 05 的 Transformer 从“读书”搬到“看图”——图切成字（ViT）→ 图文对上（CLIP）→ 图接大脑说话（LLaVA toy）。